In [1]:
import os
from pathlib import Path

# Load HF_TOKEN from .env (lines look like `export HF_TOKEN=...`)
for line in Path('.env').read_text().splitlines():
    line = line.strip().removeprefix('export ')
    if '=' in line and not line.startswith('#'):
        key, value = line.split('=', 1)
        os.environ[key.strip()] = value.strip().strip('"\'')

from datasets import load_dataset

dataset = load_dataset(
    "HuggingFaceFW/fineweb",
    "CC-MAIN-2025-08",
    split="train",
    streaming=True,
)

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/250 [00:00<?, ?it/s]

In [2]:
NUM_DOCS = 1000  # the stream is huge, so only take a sample

vespa_feed = []

for doc in dataset.take(NUM_DOCS):
    # FineWeb ids look like '<urn:uuid:d66bc6fe-...>'; strip the wrapper so it is a clean Vespa doc id
    doc_id = doc['id'].removeprefix('<urn:uuid:').removesuffix('>')
    vespa_feed.append({
        'id': doc_id,
        'fields': {
            'id': doc_id,
            'url': doc['url'],
            'text': doc['text'],
            'dump': doc['dump'],
            'year': int(doc['date'][:4]),  # date is e.g. '2013-05-18T05:48:54Z'
            'token_count': doc['token_count'],
        }
    })

print(len(vespa_feed))
print(vespa_feed[0]['fields']['url'])

1000
http://595tz744.cc/the-business-of-fashion-education-courses-crafting-the-leaders-of-tomorrow/


In [3]:
from vespa.application import ApplicationPackage

app_package = ApplicationPackage(name='fineweb')

In [4]:
from vespa.package import Schema, Document

film_schema = Schema(name='fineweb_schema', document=Document())

In [5]:
from vespa.package import Field

film_schema.add_fields(
    Field(
        name='url',
        type='string',
        indexing=['index', 'summary'],
        index="enable-bm25",
    ),
    Field(
        name='text',
        type='string',
        indexing=['index', 'summary'],
        index="enable-bm25",
    ),
    Field(
        name='dump',
        type='string',
        indexing=['attribute', 'summary']
    ),
    Field(
        name='year',
        type='int',
        indexing=['attribute', 'summary']
    ),
    Field(
        name='token_count',
        type='int',
        indexing=['attribute', 'summary']
    ),
    Field(
        name='id',
        type='string',
        indexing=['summary']
    )
)

In [6]:
from vespa.package import FieldSet, RankProfile

film_schema.add_field_set(
    FieldSet(
        name='default',
        fields=['url', 'text']
    ))

# Rank on both fields, boosting url matches a bit
film_schema.add_rank_profile(
    RankProfile(
        name='default',
        first_phase='bm25(text) + 2 * bm25(url)'
    ))

In [7]:
app_package.add_schema(film_schema)

In [9]:
from vespa.deployment import VespaCloud

vespa_cloud = VespaCloud(
    tenant='akashsinghal',
    application='fineweb',
    application_package=app_package
)

Setting application...
Running: vespa config set application akashsinghal.fineweb.default
Setting target cloud...
Running: vespa config set target cloud
Success: set target to cloud in global config at /Users/akashsinghal/.vespa/config.yaml

No api-key found for control plane access. Using access token.
No auth.json found. Please authenticate.
Your Device Confirmation code is: JRLB-GHFB
Automatically open confirmation page in your default browser? [Y/n] Y
Opened link in your browser:
	 https://login.console.vespa-cloud.com/activate?user_code=JRLB-GHFB
Waiting for login to complete in browser ... done;1m⣻
Success: Logged in
 auth.json created at /Users/akashsinghal/.vespa/auth.json
Successfully obtained access token for control plane access.
Certificate and key not found in /Users/akashsinghal/Sites/noon/search/.vespa or /Users/akashsinghal/.vespa/akashsinghal.fineweb.default: Creating new cert/key pair with vespa CLI.
Generating certificate and key...
Running: vespa auth cert -N
Succes

In [10]:
app = vespa_cloud.deploy(region="aws-euw1-az1")

Deployment started in run 1 of dev-aws-euw1-az1 for akashsinghal.fineweb.default. This may take a few minutes the first time.
INFO    [07:03:01]  Deploying platform version 8.766.27 and application dev build 1 for dev-aws-euw1-az1 of default ...
INFO    [07:03:01]  Using CA signed certificate version 1
INFO    [07:03:04]  Using 1 nodes in container cluster 'fineweb_container'
INFO    [07:03:07]  Using 1 nodes in container cluster 'fineweb_container'
INFO    [07:03:14]  Session 8 for tenant 'akashsinghal' prepared and activated.
INFO    [07:03:44]  ######## Details for all nodes ########
INFO    [07:03:44]  h13418a.dev.euw1-az1.aws.vespa-cloud.net: expected to be UP
INFO    [07:03:44]  --- platform vespaai/cloud-tenant:8.766.27
INFO    [07:03:44]  --- container on port 4080 has not started 
INFO    [07:03:44]  --- metricsproxy-container on port 19092 has not started 
INFO    [07:03:44]  h3290h.dev.euw1-az1.aws.vespa-cloud.net: expected to be UP
INFO    [07:03:44]  --- platform vespaai/c

In [11]:
cert_path = app.cert
key_path = app.key

endpoint = vespa_cloud.get_mtls_endpoint(region="aws-euw1-az1")

Found mtls endpoint for fineweb_container
URL: https://b4cdc477.a5b49224.z.vespa-app.cloud/


In [12]:
from vespa.application import Vespa

vespa_application = Vespa(endpoint, cert=cert_path, key=key_path)

In [13]:
from vespa.io import VespaResponse

def callback(response: VespaResponse, id:str):
    if not response.is_successful():
        print(f"Error when feeding document {id}: {response.get_json()}")

vespa_application.feed_iterable(vespa_feed, schema='fineweb_schema', callback=callback)

In [14]:
from vespa.io import VespaQueryResponse

def print_hits(response: VespaQueryResponse):
    for i, hit in enumerate(response.hits):
        f = hit['fields']
        print(f'{i+1:>3}  {f["id"][:8]}  {f["year"]}  {f["token_count"]:>6} tok  {f["url"]}')

In [31]:
# Searching for 'tornado' in the default fieldset (url + text)

response = vespa_application.query(
    yql="select * from sources * where text contains 'usa'", # The default limit in Vespa is 10 results
)

print_hits(response)

  1  adf04222  2025    1722 tok  https://getassignmenthelponline.com/us/
  2  03c52d8d  2025     257 tok  https://comfify.com/products/modern-mail-sorter-organizer-for-wall-with-3-double-key-hooks-wooden-wall-mount-mail-holder-organizer-modern-farmhouse-wall-decor-for-entryway-made-of-paulownia-wood-white
  3  88ed6153  2025     488 tok  https://9to5carwallpapers.com/2017-suzuki-jimny-redesign-and-up-gradation-2017/
  4  bf3015de  2025     158 tok  https://abviris.de/abviris-and-mayo-clinic-sign-cooperation-for-research-into-hpv-induced-cancer-diagnostics/
  5  facec844  2025     197 tok  https://hccinc.com/?URL=https%3A%2F%2Fpensiuneacoral.ro%2Ffr.php%3Fcid%3D30%26kys%3Djupe%20pepe%20jeans%20femme%26g%3D9
  6  60e4adb4  2025     205 tok  https://cut-n-rust.com/collections/all-products/products/sew-what-magnet-board
  7  78af2c95  2025     406 tok  https://bulkbookstore.com/rose-says-good-night-9780593969403
  8  6edb27c0  2025     420 tok  http://www.fatimaguide.com/useful-information

In [19]:
# Using userQuery(): the query text is passed as a separate parameter

response = vespa_application.query(
    yql="select * from sources * where userQuery() limit 100",
    query="tornado damage",
)

print_hits(response)

  1  c3aaf3f1  2025    2127 tok  https://furnishmycrib.co.uk/terms-conditions
  2  d7e18021  2025     538 tok  http://www.healthocity.us/the-10-best-resources-for-18/
  3  670c3955  2025     773 tok  https://blog-fruit-vegetable-ipm.extension.umn.edu/2023/07/japanese-beetle-fruit-crops-mostly-good.html
  4  99476645  2025    1290 tok  https://bycloudia.life/es/products/sterling-silver-cuff-bracelet-with-detailed-lattice-work
  5  4ba55192  2025     148 tok  https://gundamplacestore.com/products/re-100-gp04g-gerbera-water-decal
  6  55561115  2025    1781 tok  https://aingerroofing.ca/barrie-roofer-roofing-siding-attic-insulation/
  7  4aa11a5f  2025    2440 tok  https://carpetcleaningfortdodge.com/2020/11/how-to-prepare-your-home-for-winter-weather/
  8  67b290f2  2025     571 tok  http://rcplastering.co.uk/
  9  67777c70  2025     311 tok  https://fiberreinforcedpolymer.com/tag/frp-for-disaster-resilience-for-space-exploration-conference-2024/
 10  ce26ef07  2025     328 tok  https://

In [ ]:
# Search by text: matches words in the url or the body
response = vespa_application.query(
    yql="select * from sources * where userQuery() limit 10",
    query="tornado damage",
)

print_hits(response)

In [ ]:
# Search by url: pass a full url or a fragment; punctuation is tokenized away
response = vespa_application.query(
    yql="select * from sources * where userQuery() limit 10",
    query="http://ap.org/Content/Press-Release/2012/How-AP-reported-in-all-formats-from-tornado-stricken-regions",
    type="any",  # match any term, ranked by how many match
)

print_hits(response)

In [ ]:
# Restrict to one field when you only want url or only text
response = vespa_application.query(
    yql="select * from sources * where url contains 'ap' and text contains 'tornado' limit 10",
)

print_hits(response)